In [1]:
import { PGlite } from "npm:@electric-sql/pglite";
import { pg_trgm } from "npm:@electric-sql/pglite/contrib/pg_trgm";
import { unaccent } from "npm:@electric-sql/pglite/contrib/unaccent";

const client = new PGlite(
  process.env.test ? "memory://" : "./../../web/.data/database",
  {
    extensions: { pg_trgm, unaccent },
  },
);


In [2]:
import { cert, getApp, getApps, initializeApp } from "npm:firebase-admin/app";
import { getFirestore } from "npm:firebase-admin/firestore";

const serviceAccountText = await Deno.readTextFile("./../env/firebase.json");
const serviceAccount = JSON.parse(serviceAccountText);

const app = getApps().length === 0
  ? initializeApp({ credential: cert(serviceAccount) })
  : getApp();

const firestore = getFirestore(app);


In [3]:
const clubsCollection = firestore.collection("clubs");
const clubsSnapshot = await clubsCollection.get();


In [4]:
import { stringify } from "jsr:@std/csv";

const clubsData = clubsSnapshot.docs.map((doc) => ({
  uid: doc.id,
  name: doc.data().name,
  isActive: doc.data().active,
  league: doc.data().zdp ? "junior" : "senior",
  region: null,
}));

const csvData = stringify(clubsData, {
  columns: ["uid", "name", "isActive", "league", "region"],
});

await Deno.mkdir("./../data", { recursive: true });
await Deno.writeTextFile("./../data/clubs.csv", csvData);


In [5]:
const usersCollection = firestore.collection("users");
// Get the data sorted by createdAt in ascending order
const usersSnapshot = await usersCollection.orderBy("createdAt", "asc").get();


In [6]:
import { stringify } from "jsr:@std/csv";

const usersData = usersSnapshot.docs.map((doc) => {
  const data = doc.data();

  return {
    uid: doc.id,
    name: data.name ?? "",
    surname: data.surname ?? "",
    role: data.role ?? "user",
    email: data.email ?? "",
    phone: data.phone ?? "",
    birthDate: data.birthdate?.toDate()?.toISOString().split("T")[0] ?? "",
    createdAt: data.createdAt?.toDate()?.toISOString() ?? "",
    clubId: data.club?.id ?? "",
    seasons: data.seasons
      ? data.seasons.map((season: { year: number | string }) => season.year)
        .join(";")
      : "",
    clubManager: data.clubManager ?? false,
    address: data.address ?? "",
    streetAddress: "",
    postalCode: data.postalCode ?? "",
    city: data.city ?? "",
  };
});

const usersCsvData = stringify(usersData, {
  columns: [
    "uid",
    "name",
    "surname",
    "role",
    "email",
    "phone",
    "birthDate",
    "createdAt",
    "clubId",
    "seasons",
    "clubManager",
    "address",
    "streetAddress",
    "postalCode",
    "city",
  ],
});

await Deno.writeTextFile("./../data/users.csv", usersCsvData);


In [7]:
const hashesCollection = firestore.collection("hashes");
const hashesSnapshot = await hashesCollection.get();


In [8]:
// Go through each document in the hashes collection and if the group them by the email field. Keep only the latest document for each email (createdAt field).

const hashesData = hashesSnapshot.docs.map((doc) => ({
  uid: doc.id,
  email: doc.data().email ?? "",
  hash: doc.data().hash ?? "",
  createdAt: doc.data().createdAt?.toDate()?.toISOString() ?? "",
}));

const latestHashesMap = new Map<string, any>();

for (const hash of hashesData) {
  const existingHash = latestHashesMap.get(hash.email);

  if (
    !existingHash || new Date(hash.createdAt) > new Date(existingHash.createdAt)
  ) {
    latestHashesMap.set(hash.email, hash);
  }
}

const latestHashesData = Array.from(latestHashesMap.values());


In [10]:
// Read the file clubs_x.csv
import { parse } from "jsr:@std/csv";

type ClubX = {
  id: number | null;
  uid: string;
  name: string;
  isActive: boolean;
  league: "junior" | "senior" | "university" | null;
  region: "western" | "central" | "eastern" | null;
};

const clubsXFile = await Deno.readTextFile("./../data/clubs_x.csv");
const clubsXData: ClubX[] = await parse(clubsXFile, {
  skipFirstRow: true,
  strip: true,
  columns: ["uid", "name", "isActive", "league", "region"],
});

// Add the clubxdata to the pglite client and map generated ids to the clubs x  using returning data.
clubsXData.forEach(async (club) => {
  await client.query(
    "INSERT INTO clubs (name, is_active, league, region) VALUES ($1, $2, $3, $4) RETURNING id",
    [club.name, club.isActive, club.league, club.region],
  ).then((result) => {
    const generatedId = result.rows[0].id;
    clubsXData.find((c) => c.uid === club.uid)!.id = generatedId;
    console.log(`Inserted club ${club.name} with generated id ${generatedId}`);
  }).catch((error) => {
    console.error(`Error inserting club ${club.name}:`, error);
  });
});


Inserted club Sučany with generated id 1
Inserted club SZŠ Félix v Źiline with generated id 2
Inserted club Univerzita sv. Cyrila a Metoda v Trnave with generated id 3
Inserted club Súkromná spojená škola, M.Falešníka with generated id 4
Inserted club ZŠ Bartolomeja Krpelca, Bardejov with generated id 5
Inserted club Gym. Varšavská cesta (mladší) with generated id 6
Inserted club ZŠ Martinská, Žilina with generated id 7
Inserted club ŠPMNDaG (mladší) with generated id 8
Inserted club Súkromné gym. Katkin Park with generated id 9
Inserted club Bratislavský debatný spolok with generated id 10
Inserted club Základná Škola u Filipa with generated id 11
Inserted club Marguškin Andrejov a Jozefov Absolventský Klub with generated id 12
Inserted club Gymnázium Andreja Sládkoviča with generated id 13
Inserted club Gym. J. G. Tajovského (Taják) with generated id 14
Inserted club Gym. Leonarda Stockela with generated id 15
Inserted club SPŠE Prešov with generated id 16
Inserted club OA a SOŠ obch

In [ ]:
type UserX = {
  id: number | null;
  uid: string;
  name: string;
  surname: string;
  role: "user" | "admin" | "superadmin" | null;
  email: string;
  phone: string;
  birthDate: string | null;
  createdAt: string | null;
  clubId: number | null;
  seasons: string | null;
  clubManager: boolean | null;
  address: string | null;
  streetAddress: string | null;
  postalCode: string | null;
  city: string | null;
};

// const usersXData: UserX[] = await parse("./../data/users_x.csv");
const usersXFile = await Deno.readTextFile("./../data/users_x.csv");
const usersXData: UserX[] = await parse(usersXFile, {
  skipFirstRow: true,
  strip: true,
  columns: [
    "uid",
    "name",
    "surname",
    "role",
    "email",
    "phone",
    "birthDate",
    "createdAt",
    "clubId",
    "seasons",
    "clubManager",
    "address",
    "streetAddress",
    "postalCode",
    "city",
  ],
});

// Go trough each user in the usersXData and if the user has role "coach", change it to "user" and set the clubManager field to true.
usersXData.forEach((user) => {
  if (user.role === "coach") {
    user.role = "user";
    user.clubManager = true;
  }
});

// In usersXData, replace all the empty strings with null values.
usersXData.forEach((user) => {
  Object.keys(user).forEach((key) => {
    if (user[key as keyof UserX] === "") {
      user[key as keyof UserX] = null;
    }
  });
});

const removedEmails = new Set<string>();

// Remove all users from usersXData that have name or surname or email as null.
usersXData.forEach((user, index) => {
  if (!user.name || !user.surname || !user.email) {
    console.log(
      `Removing user with uid ${user.uid} because of missing name, surname or email`,
    );
    removedEmails.add(user.email);
    usersXData.splice(index, 1);
  }
});

// Remove all users from usersXData that have birthdate and address as null.
usersXData.forEach((user, index) => {
  if (!user.birthDate && !user.address) {
    console.log(
      `Removing user with uid ${user.uid} because of missing birthdate and address`,
    );
    removedEmails.add(user.email);
    usersXData.splice(index, 1);
  }
});

// Add the usersxdata to the pglite client and map generated ids to the users x using returning data.
usersXData.forEach(async (user) => {
  let generatedId: number | null = null;

  await client.query(
    "INSERT INTO users (name, surname, role, email, phone, birth_date, created_at, street, postal_code, town, email_verified) VALUES ($1, $2, $3, $4, $5, $6, $7, $8, $9, $10, true) RETURNING id",
    [
      user.name,
      user.surname,
      user.role,
      user.email,
      user.phone,
      user.birthDate,
      user.createdAt,
      user.streetAddress,
      user.postalCode,
      user.city,
    ],
  ).then((result) => {
    generatedId = result.rows[0].id;
    usersXData.find((u) => u.uid === user.uid)!.id = generatedId;
    console.log(
      `Inserted user ${user.name} ${user.surname} with generated id ${generatedId}`,
    );
  }).catch((error) => {
    generatedId = null;
    removedEmails.add(user.email);
    console.error(`Error inserting user ${user.name} ${user.surname}:`, error);
  });

  console.log(
    `Processing user ${user.name} ${user.surname} with clubId ${user.clubId} and seasons ${user.seasons}. The generated id is ${generatedId}`,
  );

  // If the user has a clubId, for each season separated by ;, insert a row into the club_memberships table with the generated user id, the club id, and the season year.
  if (user.clubId && user.seasons && generatedId) {
    const seasons = user.seasons.split(";");
    seasons.forEach(async (season) => {
      const mappedClubId = clubsXData.find((c) => c.uid === user.clubId)?.id;
      if (!mappedClubId) {
        console.error(
          `Error: Could not find mapped club id for user ${user.name} ${user.surname} with club uid ${user.clubId}`,
        );
        return;
      }

      // If user is younger than 14 years old in the given season, set the registration_type to "junior_student", < 19 "senior)student", < 30 "graduate" >= 30 "teacher"
      const birthYear = new Date(user.birthDate ?? "").getFullYear();
      const seasonYear = parseInt(season);
      let registrationType:
        | "junior_student"
        | "senior_student"
        | "graduate"
        | "teacher" = "teacher";

      if (birthYear && seasonYear) {
        const age = seasonYear - birthYear;
        if (age < 14) {
          registrationType = "junior_student";
        } else if (age < 19) {
          registrationType = "senior_student";
        } else if (age < 30) {
          registrationType = "graduate";
        } else {
          registrationType = "teacher";
        }
      }

      await client.query(
        "INSERT INTO club_memberships (user_id, club_id, season, confirmed, registration_type) VALUES ($1, $2, $3, true, $4)",
        [generatedId, mappedClubId, season, registrationType],
      ).then(() => {
        console.log(
          `Inserted club membership for user ${user.name} ${user.surname} in club ${user.clubId} for season ${season}`,
        );
      }).catch((error) => {
        console.error(
          `Error inserting club membership for user ${user.name} ${user.surname} in club ${user.clubId} for season ${season}:`,
          error,
        );
      });
    });
  }

  // If clubManager is true, insert a row into the club_managers table with the generated user id and the mapped club id.
  if (user.clubManager && user.clubId && generatedId !== null) {
    const mappedClubId = clubsXData.find((c) => c.uid === user.clubId)?.id;
    if (!mappedClubId) {
      console.error(
        `Error: Could not find mapped club id for user ${user.name} ${user.surname} with club uid ${user.clubId}`,
      );
      return;
    }
    await client.query(
      "INSERT INTO club_managers (user_id, club_id) VALUES ($1, $2)",
      [generatedId, mappedClubId],
    ).then(() => {
      console.log(
        `Inserted club manager for user ${user.name} ${user.surname} in club ${user.clubId}`,
      );
    }).catch((error) => {
      console.error(
        `Error inserting club manager for user ${user.name} ${user.surname} in club ${user.clubId}:`,
        error,
      );
    });
  }

  // If the user has a hash, insert a row into the accounts table with the generated user id and the hash.
  const userHash = latestHashesData.find((h) => h.email === user.email);
  if (userHash && generatedId !== null) {
    await client.query(
      "INSERT INTO accounts (user_id, provider_id, issuer, account_id, password) VALUES ($1, 'credential', 'local:credential', $1, $2)",
      [generatedId, userHash.hash],
    ).then(() => {
      console.log(
        `Inserted account for user ${user.name} ${user.surname}`,
      );
    }).catch((error) => {
      console.error(
        `Error inserting account for user ${user.name} ${user.surname}:`,
        error,
      );
    });
  }

  // In usersSnapshot, check if user has supervisor and supervisorEmail fields. If so, insert a row into the legal_guardians table with the generated user id, the supervisor name, and the supervisor email.
  const userSnapshot = usersSnapshot.docs.find((doc) => doc.id === user.uid);
  if (userSnapshot) {
    const userData = userSnapshot.data();
    if (
      userData.supervisor && userData.supervisorEmail && generatedId !== null
    ) {
      await client.query(
        "INSERT INTO legal_guardians (user_id, name, email) VALUES ($1, $2, $3)",
        [generatedId, userData.supervisor, userData.supervisorEmail],
      ).then(() => {
        console.log(
          `Inserted legal guardian for user ${user.name} ${user.surname}`,
        );
      }).catch((error) => {
        console.error(
          `Error inserting legal guardian for user ${user.name} ${user.surname}:`,
          error,
        );
      });
    }
  }
});

// Write the removed emails to a file removed_emails.txt
await Deno.writeTextFile(
  "./../data/removed_emails.txt",
  Array.from(removedEmails).join("\n"),
);


Removing user with uid tZfWvnwmY5f16M6V7IacWcmiz4r2 because of missing name, surname or email
Removing user with uid UEsELg5yT7aNgvUegQLH70UkMTt2 because of missing name, surname or email
Removing user with uid BJ8yRO2DyhZPvzd02TpBIqykF2X2 because of missing name, surname or email
Removing user with uid yrwjj6KCRpTIHgyrR7rrtfCbLVD3 because of missing name, surname or email
Removing user with uid EloGr7T9bFPbxUdWGIBx2aw3v133 because of missing name, surname or email
Removing user with uid VNegjIW2G0hl8d0w9uhXq1mXriv1 because of missing name, surname or email
Removing user with uid TyQ8bu4ig8biD94TL5tdb3IRs0q2 because of missing name, surname or email
Removing user with uid egV1NyMgvQYbostdguBDrzu9fhu2 because of missing name, surname or email
Removing user with uid UzfJ49138uacN5WEofTIjdP2Rmu1 because of missing name, surname or email
Removing user with uid lleVwPOQH0PPuItVduLdKcn4z692 because of missing name, surname or email
Removing user with uid oqDEtwc3uBelSxmGrlEUzlSkHw32 because 

Processing user Petra Bujňáková with clubId kjORYUEeZKTLVqWPWGYP and seasons 2024;2025;2026;2025;2026. The generated id is null
Processing user Ester  Molitorisová with clubId kjORYUEeZKTLVqWPWGYP and seasons 2024;2025;2025;2026. The generated id is null
Processing user Hana Hagarová with clubId kjORYUEeZKTLVqWPWGYP and seasons 2024;2025. The generated id is null
Processing user Andrea Kotrčová with clubId tOn6oeCGdjT1Zx78cJkf and seasons 2024;2025;2026;2025;2026. The generated id is null
Processing user Tomáš Bačo with clubId 69yGA89wrEErLdElbQiP and seasons 2024;2025. The generated id is null
Processing user Karolína Šofranková with clubId du02xtXBfez8nGcMkx8i and seasons 2024;2025;2025;2026. The generated id is null
Processing user Matej Chriašteľ with clubId QnLemmf5qhClABxwW6YH and seasons 2024;2025. The generated id is null
Processing user Ema Donátová with clubId 00a1bjaC42CX321dnJU1 and seasons 2024;2025;2025;2026. The generated id is null
Processing user Alexej Sopčák with clu

Error inserting user Branislav Juhás: error: relation "users" does not exist
    at pe.Ue (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:442:11)
    at pe.tt (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:220:16)
    at pe.parse (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:104:21)
    at N.qe (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/index.js:753:26)
    at file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/index.js:706:14
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x5a5ccf)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x27eaeb)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x2869d1)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x286c2d)
    at <anonymous> (wasm://wasm/0267bb86:wasm

In [ ]:
// Close the pglite client
await client.close();
